# Perceptron: Do Neurônio ao PyTorch

> *"A máquina que aprende com seus próprios erros"*

Este notebook é um guia interativo sobre o Perceptron — a unidade fundamental das redes neurais — desde a biologia até a implementação em PyTorch.

---

## 📚 Assuntos abordados

| Conceito                     | O que é                                                 |
|------------------------------|---------------------------------------------------------|
| **Sinapses**                 | Como neurônios lidam com dados                          |
| **Soma Ponderada**           | O cálculo essencial do Perceptron                       |
| **Ativação**                 | A decisão de "disparar ou não"                          |
| **Predição**                 | Como gerar uma saída                                    |
| **Treinamento**              | Como o Perceptron aprende (indução do modelo preditivo) |
| **Função de Erro**           | Como medir o erro                                       |
| **Mínimos Locais e Globais** | O problema da otimização                                |

---


## 🔗 1. Sinapses — As Conexões do Neurônio

Na biologia, um neurônio recebe sinais elétricos de outros neurônios através das *sinapses*.  
Cada sinapse tem uma intensidade — algumas conexões são mais importantes que outras.

No Perceptron, essa ideia se traduz em:

- *Entradas* → os sinais que chegam (`x₁, x₂, ..., xₙ`)  
- *Pesos* → a força de cada conexão (`w₁, w₂, ..., wₙ`)  
- *Intercepto (bias)* → um limiar de ativação (`b`)

```
x₁ ──(w₁)──┐
x₂ ──(w₂)──┤──[Σ]──[f]──► ŷ
x₃ ──(w₃)──┘
          ↑
         intercepto (b)
```

> 💡 **Analogia**: Os pesos funcionam como a rodinha de volume de um aparelho de som para cada entrada. Um peso alto significa que aquela entrada tem maior influência na decisão final. Nota: _Uma diferença aqui, é que o peso pode ser negativo (inibitório, se colocar em termos biológicos)_.


In [ ]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

# Configurações de visualização
plt.rcParams['figure.facecolor'] = '#0d1117'
plt.rcParams['axes.facecolor'] = '#161b22'
plt.rcParams['axes.edgecolor'] = '#30363d'
plt.rcParams['text.color'] = '#e6edf3'
plt.rcParams['axes.labelcolor'] = '#e6edf3'
plt.rcParams['xtick.color'] = '#8b949e'
plt.rcParams['ytick.color'] = '#8b949e'
plt.rcParams['grid.color'] = '#21262d'

print("✅ Pacotes carregados com sucesso!")
print(f"   PyTorch versão: {torch.__version__}")

# ─── Representação das sinapses ───────────────────────────────────────────────
torch.manual_seed(42)

# 3 entradas (features)
x = torch.tensor([1.5, -0.8, 2.1])

# Pesos inicializados aleatoriamente (simulando sinapses)
pesos = torch.tensor([0.6, -0.3, 0.9])

# Intercepto
intercepto = torch.tensor(0.2)

print("\n🔗 Sinapses do Perceptron:")
print(f"   Entradas     x = {x.numpy()}")
print(f"   Pesos        w = {pesos.numpy()}")
print(f"   Intercepto   b = {intercepto.item():.2f}")
print("\nCada peso controla o quanto cada entrada influencia a saída.")  # Podemos considerar que o intercepto é um peso com entrada sempre 1.


## ➕ 2. Soma Ponderada

A *soma ponderada* é o coração matemático do Perceptron. Representa um hiperplano que divide o espaço em duas metades: onde ficam instâncias de classe **+** ou classe **-**.
Cada entrada é multiplicada pelo seu peso e todos os resultados são somados com o intercepto:

$$z = w_1 x_1 + w_2 x_2 + \cdots + w_n x_n + b = \mathbf{w}^T \mathbf{x} + b$$

Essa operação é equivalente a um produto escalar *dot product* no espaço vetorial.



In [ ]:
# ─── Soma Ponderada ─────────────────────────────────────────────────────────

# Produto escalar: w · x + b
z = torch.dot(pesos, x) + intercepto

print("📐 Soma Ponderada:")
print(f"   z = ({pesos[0]:.1f} × {x[0]:.1f}) + ({pesos[1]:.1f} × {x[1]:.1f}) + ({pesos[2]:.1f} × {x[2]:.1f}) + {intercepto:.1f}")
print(f"   z = {pesos[0] * x[0]:.3f} + {pesos[1] * x[1]:.3f} + {pesos[2] * x[2]:.3f} + {intercepto:.3f}")
print(f"   z = {z.item():.4f}")

# ─── Visualização ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.set_xlim(-1, 10)
ax.set_ylim(-1, 4)
ax.axis('off')
ax.set_title('Diagrama: Soma Ponderada', color='#58a6ff', fontsize=14, pad=15)

cores_entrada = ['#ff7b72', '#79c0ff', '#7ee787']
labels_x = ['x₁ = 1.5', 'x₂ = -0.8', 'x₃ = 2.1']
labels_w = ['w₁=0.6', 'w₂=-0.3', 'w₃=0.9']

for i, (lx, lw, cor) in enumerate(zip(labels_x, labels_w, cores_entrada)):
    y_pos = 3 - i
    ax.annotate('', xy=(4.5, 1.8), xytext=(1.2, y_pos),
                arrowprops=dict(arrowstyle='->', color=cor, lw=2))
    ax.text(0.1, y_pos, lx, color=cor, fontsize=12, va='center',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#161b22', edgecolor=cor))
    ax.text(2.5, (y_pos + 1.8) / 2 + 0.15, lw, color=cor, fontsize=10, ha='center',
            style='italic')

# Nó somador
circulo = plt.Circle((5.2, 1.8), 0.5, color='#388bfd', fill=True, zorder=5)
ax.add_patch(circulo)
ax.text(5.2, 1.8, 'Σ', color='white', fontsize=16, ha='center', va='center', zorder=6, fontweight='bold')

# Intercepto
ax.annotate('', xy=(5.2, 1.3), xytext=(5.2, 0.6),
            arrowprops=dict(arrowstyle='->', color='#ffa657', lw=2))
ax.text(5.2, 0.35, 'b = 0.2', color='#ffa657', fontsize=11, ha='center',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#161b22', edgecolor='#ffa657'))

# Saída
ax.annotate('', xy=(8.0, 1.8), xytext=(5.7, 1.8),
            arrowprops=dict(arrowstyle='->', color='#e6edf3', lw=2.5))
ax.text(8.3, 1.8, f'z = {z.item():.3f}', color='#d2a8ff', fontsize=13, va='center', fontweight='bold')

plt.tight_layout()
plt.show()
print(f"\n✅ Soma ponderada calculada: z = {z.item():.4f}")


## ⚡ 3. Função de Ativação

A **soma ponderada** `z` é apenas um número contínuo. A **função de ativação** decide:

> *"Dado esse sinal `z`, qual deve ser a saída do neurônio?"*

### Funções de Ativação Clássicas

| Função            | Fórmula                 | Uso                               |
|-------------------|-------------------------|-----------------------------------|
| **Degrau** (Step) | `1 se z ≥ 0, senão 0`   | Perceptron original               |
| **Sigmoid**       | `1 / (1 + e⁻ᶻ)`         | Classificação binária             |
| **ReLU**          | `max(0, z)`             | Redes profundas (default moderno) |
| **Tanh**          | `(eᶻ - e⁻ᶻ)/(eᶻ + e⁻ᶻ)` | Saída entre -1 e 1                |

A função de ativação introduz **não-linearidade** — sem ela, qualquer rede neural (por mais camadas que tenha) se reduziria a uma única transformação linear, limitando drasticamente o que ela pode aprender.


In [ ]:
# ─── Funções de Ativação ────────────────────────────────────────────────────

z_range = torch.linspace(-6, 6, 300)

# Calculando as funções
step = (z_range >= 0).float()
sigmoid = torch.sigmoid(z_range)
relu = torch.relu(z_range)
tanh = torch.tanh(z_range)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Funções de Ativação', color='#58a6ff', fontsize=16, fontweight='bold')

configs = [
    (axes[0, 0], step, '#ff7b72', 'Degrau (Step)', 'Perceptron original\nSaída: {0, 1}'),
    (axes[0, 1], sigmoid, '#79c0ff', 'Sigmoid', 'Classificação binária\nSaída: (0, 1)'),
    (axes[1, 0], relu, '#7ee787', 'ReLU', 'Redes profundas modernas\nSaída: [0, ∞)'),
    (axes[1, 1], tanh, '#ffa657', 'Tanh', 'Saída centrada em zero\nSaída: (-1, 1)'),
]

for ax, y, cor, titulo, desc in configs:
    ax.plot(z_range.numpy(), y.numpy(), color=cor, lw=2.5)
    ax.axhline(0, color='#30363d', lw=1)
    ax.axvline(0, color='#30363d', lw=1)
    ax.set_title(titulo, color=cor, fontsize=12, fontweight='bold')
    ax.set_xlabel('z (entrada)', fontsize=10)
    ax.set_ylabel('f(z) (saída)', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.text(0.97, 0.05, desc, transform=ax.transAxes, fontsize=8.5,
            va='bottom', ha='right', color='#8b949e',
            bbox=dict(boxstyle='round', facecolor='#0d1117', alpha=0.7))

    # Marca o ponto atual z
    z_val = z.item()
    y_val = torch.sigmoid(z) if 'Sigmoid' in titulo else torch.relu(z) if 'ReLU' in titulo else torch.tanh(z) if 'Tanh' in titulo else torch.tensor(1.0 if z_val >= 0 else 0.0)
    ax.scatter([z_val], [y_val.item()], color='white', s=80, zorder=5)
    ax.annotate(f'  z={z_val:.2f} f={y_val.item():.2f}', xy=(z_val, y_val.item()), color='white', fontsize=8.5)

plt.tight_layout()
plt.show()

# PyTorch: aplicando ativação ao nosso z
print("\n⚡ Aplicando ativações ao nosso z = {:.4f}:".format(z.item()))
print(f"   Degrau  → {(z >= 0).float().item():.0f}")
print(f"   Sigmoid → {torch.sigmoid(z).item():.4f}")
print(f"   ReLU    → {torch.relu(z).item():.4f}")
print(f"   Tanh    → {torch.tanh(z).item():.4f}")


## 🎯 4. Predição (*Forward Pass*)

A **predição** é o processo de combinar tudo o que vimos:

$$\hat{y} = f\left(\sum_{i=1}^{n} w_i x_i + \right) = f(\mathbf{w}^T\mathbf{x} + b)$$

Esse fluxo de **entrada → soma ponderada → ativação → saída** é chamado de *forward pass*.

Em PyTorch, usamos `nn.Linear` para a parte linear e aplicamos a ativação manualmente ou via `nn.Sequential`.


In [ ]:
# ─── Predição com PyTorch ───────────────────────────────────────────────────

# Problema: classificar pontos em 2D (AND lógico)
# x1, x2 → 0 ou 1 (AND: saída 1 apenas quando ambos são 1)

X = torch.tensor([[0., 0.],
                  [0., 1.],
                  [1., 0.],
                  [1., 1.]])

y_and = torch.tensor([[0.], [0.], [0.], [1.]])

print("🎯 Problema: AND Lógico")
print("─" * 35)
print(f"{'x1':>4} {'x2':>4} {'y (AND)':>10}")
print("─" * 35)
for xi, yi in zip(X, y_and):
    print(f"{xi[0].item():>4.0f} {xi[1].item():>4.0f} {yi.item():>10.0f}")


# ─── Definindo o Perceptron em PyTorch ─────────────────────────────────────
class Perceptron(nn.Module):
    def __init__(self, n_entradas):
        super().__init__()
        # nn.Linear(in, out) cria: w (pesos) + b (intercepto) automaticamente
        self.linear = nn.Linear(n_entradas, 1)
        self.ativacao = nn.Sigmoid()

    def forward(self, x_):
        z_ = self.linear(x_)  # Soma ponderada
        y_hat = self.ativacao(z_)  # Ativação
        return y_hat


torch.manual_seed(0)
modelo = Perceptron(n_entradas=2)

print("\n🧠 Perceptron criado:")
print(f"   Pesos iniciais : {modelo.linear.weight.data.numpy()}")
print(f"   Intercepto inicial   : {modelo.linear.intercepto.data.numpy()}")

# Predições ANTES do treinamento
with torch.no_grad():
    predicoes_antes = modelo(X)

print("\n📊 Predições ANTES do treinamento:")
print("─" * 40)
print(f"{'x1':>4} {'x2':>4} {'Esperado':>10} {'Predito':>10}")
print("─" * 40)
for xi, yi, pi in zip(X, y_and, predicoes_antes):
    print(f"{xi[0].item():>4.0f} {xi[1].item():>4.0f} {yi.item():>10.0f} {pi.item():>10.4f}")
print("\n⚠️  As predições estão erradas — nada foi aprendido!")


## 🏋️ 5. Treinamento — Como o Perceptron Aprende

O treinamento é o processo de **ajustar os pesos** para que as predições fiquem cada vez mais próximas dos valores reais.

### O Loop de Treinamento

```
Para cada época (epoch):
    1. Forward Pass → calcula ŷ
    2. Calcula o erro → L = loss(ŷ, y)
    3. Backward Pass → calcula gradientes ∂L/∂w
    4. Atualiza pesos → w = w - lr × ∂L/∂w
```

### Gradiente Descendente

$$w \leftarrow w - \eta \cdot rac{\partial L}{\partial w}$$

Onde `η` (eta) é a taxa de aprendizado (_learning rate_) — controla o tamanho do passo na direção que reduz o erro.

> 🧭 **Analogia**: Imagine que você está numa montanha com neblina e quer descer ao vale. Você não consegue ver o caminho inteiro, mas pode sentir a inclinação do terreno sob seus pés (gradiente) e dá um passo na direção de maior descida. O learning rate é o tamanho desse passo.


In [ ]:
# ─── Treinamento do Perceptron ───────────────────────────────────────────────

torch.manual_seed(0)
modelo = Perceptron(n_entradas=2)

# Função de perda: Binary Cross-Entropy (adequada para classificação binária)
criterio = nn.BCELoss()

# Otimizador: Gradiente Descendente Estocástico
otimizador = torch.optim.SGD(modelo.parameters(), lr=0.5)

historico_loss = []
historico_pesos = []
n_epocas = 200

print("🏋️  Iniciando treinamento...")
print(f"   Épocas       : {n_epocas}")
print(f"   Learning Rate: 0.5")
print(f"   Função de Perda: BCE Loss\n")

for epoca in range(n_epocas):
    # 1. Forward pass
    y_pred = modelo(X)

    # 2. Calcula o erro
    loss = criterio(y_pred, y_and)

    # 3. Zera gradientes anteriores (importante no PyTorch!)
    otimizador.zero_grad()

    # 4. Backward pass (calcula gradientes)
    loss.backward()

    # 5. Atualiza pesos
    otimizador.step()

    historico_loss.append(loss.item())
    historico_pesos.append(modelo.linear.weight.data.clone().numpy().flatten())

    if (epoca + 1) % 40 == 0:
        print(f"   Época {epoca + 1:>3} | Loss: {loss.item():.6f} | "
              f"Pesos: [{modelo.linear.weight.data[0, 0].item():+.3f}, "
              f"{modelo.linear.weight.data[0, 1].item():+.3f}] | "
              f"Intercepto: {modelo.linear.intercepto.data.item():+.3f}")

print("\n✅ Treinamento concluído!")

# ─── Visualizando a queda do loss ────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Progresso do Treinamento', color='#58a6ff', fontsize=15, fontweight='bold')

# Loss ao longo do tempo
epocas_eixo = range(1, n_epocas + 1)
ax1.plot(epocas_eixo, historico_loss, color='#ff7b72', lw=2)
ax1.fill_between(epocas_eixo, historico_loss, alpha=0.15, color='#ff7b72')
ax1.set_xlabel('Época', fontsize=11)
ax1.set_ylabel('BCE Loss', fontsize=11)
ax1.set_title('Curva de Aprendizado', color='#e6edf3', fontsize=12)
ax1.grid(True, alpha=0.3)
# noinspection PyTypeHints
ax1.annotate(f'Loss final\n{historico_loss[-1]:.5f}', xy=(n_epocas, historico_loss[-1]),
             xytext=(n_epocas * 0.7, historico_loss[0] * 0.6), color='#7ee787', fontsize=10,
             arrowprops=dict(arrowstyle='->', color='#7ee787'))

# Evolução dos pesos
pesos_arr = np.array(historico_pesos)
ax2.plot(epocas_eixo, pesos_arr[:, 0], color='#79c0ff', lw=2, label='w₁ (peso x₁)')
ax2.plot(epocas_eixo, pesos_arr[:, 1], color='#ffa657', lw=2, label='w₂ (peso x₂)')
ax2.set_xlabel('Época', fontsize=11)
ax2.set_ylabel('Valor do Peso', fontsize=11)
ax2.set_title('Evolução dos Pesos', color='#e6edf3', fontsize=12)
ax2.legend(facecolor='#161b22', edgecolor='#30363d')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 📉 6. Função de Erro (Loss Function)

A **função de erro** (ou loss) mede **o quão erradas são as predições** do modelo.  
É ela quem guia o treinamento — o objetivo é minimizá-la.

### Funções de Erro Comuns

**MSE — Mean Squared Error** (regressão):
$$L_{MSE} = rac{1}{n}\sum_{i=1}^n (\hat{y}_i - y_i)^2$$

**BCE — Binary Cross-Entropy** (classificação binária):
$$L_{BCE} = -rac{1}{n}\sum_{i=1}^n \left[y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i) \right]$$

> 🔑 A BCE penaliza **muito fortemente** predições confiantes e erradas (ex: prever 0.99 quando a resposta é 0), o que força o modelo a ser mais cauteloso.


In [ ]:
# ─── Visualizando a Função de Erro ─────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Funções de Erro', color='#58a6ff', fontsize=15, fontweight='bold')

# ─── MSE ────────────────────────────────────────────────────────────────────
ax1, ax2 = axes
y_verdadeiro = torch.tensor(1.0)
y_pred_range = torch.linspace(0.01, 0.99, 200)

mse_vals = (y_pred_range - y_verdadeiro) ** 2
bce_vals = -(y_verdadeiro * torch.log(y_pred_range) +
             (1 - y_verdadeiro) * torch.log(1 - y_pred_range))

ax1.plot(y_pred_range, mse_vals, color='#79c0ff', lw=2.5, label='MSE (y=1)')
ax1.set_xlabel('Predição ŷ', fontsize=11)
ax1.set_ylabel('Loss', fontsize=11)
ax1.set_title('MSE Loss', color='#e6edf3', fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.legend(facecolor='#161b22')
ax1.annotate('Mínimo quando\nŷ → 1 (correto)', xy=(0.99, 0.0001), xytext=(0.6, 0.4),
             color='#7ee787', fontsize=9, arrowprops=dict(arrowstyle='->', color='#7ee787'))

ax2.plot(y_pred_range, bce_vals, color='#ff7b72', lw=2.5, label='BCE (y=1)')
y_pred_0 = torch.linspace(0.01, 0.99, 200)
bce_vals_0 = -(0 * torch.log(y_pred_0) + torch.log(1 - y_pred_0))
ax2.plot(y_pred_0, bce_vals_0, color='#ffa657', lw=2.5, label='BCE (y=0)', linestyle='--')
ax2.set_xlabel('Predição ŷ', fontsize=11)
ax2.set_ylabel('Loss', fontsize=11)
ax2.set_title('Binary Cross-Entropy Loss', color='#e6edf3', fontsize=12)
ax2.set_ylim(0, 5)
ax2.grid(True, alpha=0.3)
ax2.legend(facecolor='#161b22')
ax2.annotate('Predição confiante\ne ERRADA → loss → ∞', xy=(0.99, 4.6), xytext=(0.4, 4.2),
             color='#ff7b72', fontsize=9, arrowprops=dict(arrowstyle='->', color='#ff7b72'))

plt.tight_layout()
plt.show()

# Comparando na prática
print("📊 Comparando as funções de erro (y verdadeiro = 1):\n")
exemplos = [0.1, 0.5, 0.9, 0.99]
print(f"{'ŷ (predição)':>14} {'MSE':>10} {'BCE':>10}  Interpretação")
print("─" * 60)
for yp in exemplos:
    yp_t = torch.tensor(yp)
    mse = (yp_t - 1.0) ** 2
    bce = -(torch.log(yp_t))
    qualidade = "❌ Muito errado" if yp < 0.3 else ("⚠️  Razoável" if yp < 0.8 else "✅ Bom")
    print(f"{yp:>14.2f} {mse.item():>10.4f} {bce.item():>10.4f}  {qualidade}")


## 🏔️ 7. Mínimos Locais e Globais

O treinamento de redes neurais é um problema de **otimização**: queremos encontrar os pesos que minimizam a função de erro.

### O Problema

A superfície de erro (loss landscape) pode ter:

- **Mínimo Global** 🌍 — o ponto mais baixo de toda a superfície (solução ideal)  
- **Mínimos Locais** 🕳️ — pontos baixos, mas não os mais baixos (armadilhas)  
- **Platôs** ➡️ — regiões planas onde o gradiente ≈ 0 (progresso lento)  
- **Pontos de Sela** 🎪 — mínimo em uma direção, máximo em outra

### Por que isso importa?

O **Gradiente Descendente** segue a inclinação local — sem visão global.  
Pode ficar preso em um mínimo local sem saber que existe um vale mais profundo.

### Soluções práticas

| Técnica                      | Ideia                                         |
|------------------------------|-----------------------------------------------|
| **Momentum**                 | Mantém velocidade para ultrapassar obstáculos |
| **Adam Optimizer**           | Taxa de aprendizado adaptativa por parâmetro  |
| **Learning Rate Scheduling** | Reduz lr gradualmente                         |
| **Random Restarts**          | Treina com múltiplos pontos de início         |
| **Batch estocástico (SGD)**  | O "ruído" ajuda a escapar de mínimos locais   |


In [ ]:
# ─── Visualizando Mínimos Locais e Globais ──────────────────────────────────

fig = plt.figure(figsize=(15, 6))

# ─── 2D: Superfície de erro simplificada ─────────────────────────────────────
ax1 = fig.add_subplot(121)

w_range = np.linspace(-3, 3, 400)


# Função sintética com múltiplos mínimos
def loss_surface_1d(w):
    return 0.5 * np.sin(3 * w) + 0.3 * w ** 2 + 0.1 * np.cos(5 * w) + 0.8


loss_vals = loss_surface_1d(w_range)

ax1.plot(w_range, loss_vals, color='#79c0ff', lw=2.5)
ax1.fill_between(w_range, loss_vals, alpha=0.1, color='#79c0ff')

# Marcando mínimos locais e global
min_global_idx = np.argmin(loss_vals)
ax1.scatter(w_range[min_global_idx], loss_vals[min_global_idx], color='#7ee787', s=150, zorder=5, marker='*')
ax1.annotate('Mínimo Global ', xy=(w_range[min_global_idx], loss_vals[min_global_idx]),
             xytext=(w_range[min_global_idx] + 0.5, loss_vals[min_global_idx] + 0.3),
             color='#7ee787', fontsize=9, arrowprops=dict(arrowstyle='->', color='#7ee787'))

# Mínimos locais aproximados
for w_local, label_x in [(-2.0, -2.5), (0.5, 0.8)]:
    idx = np.argmin(np.abs(w_range - w_local))
    # busca mínimo local real na vizinhança
    janela = 30
    local_slice = loss_vals[max(0, idx - janela):idx + janela]
    local_min_idx = np.argmin(local_slice) + max(0, idx - janela)
    ax1.scatter(w_range[local_min_idx], loss_vals[local_min_idx], color='#ff7b72', s=100, zorder=5, marker='v')
    # noinspection PyTypeChecker
    ax1.annotate('Mínimo Local ', xy=(w_range[local_min_idx], loss_vals[local_min_idx]),
                 xytext=(w_range[local_min_idx] - 1.0, loss_vals[local_min_idx] + 0.35),
                 color='#ff7b72', fontsize=9, arrowprops=dict(arrowstyle='->', color='#ff7b72'))

# Simulando uma descida de gradiente que fica presa
w_start_ruim = -2.5
w_traj = [w_start_ruim]
lr_demo = 0.05
for _ in range(30):
    # noinspection PyTypeHints
    w_cur = w_traj[-1]
    grad = 0.5 * 3 * np.cos(3 * w_cur) + 0.6 * w_cur - 0.5 * np.sin(5 * w_cur)
    w_new = w_cur - lr_demo * grad
    w_traj.append(w_new)
    if abs(w_new - w_cur) < 1e-4:
        break

loss_traj = [loss_surface_1d(w) for w in w_traj]
ax1.plot(w_traj, loss_traj, color='#ffa657', lw=2, alpha=0.8, label='Trajetória presa')
# noinspection PyTypeHints
ax1.scatter(w_traj[0], loss_traj[0], color='#ffa657', s=80, marker='o', zorder=6)
# noinspection PyTypeHints
ax1.scatter(w_traj[-1], loss_traj[-1], color='#ffa657', s=100, marker='X', zorder=6)
ax1.legend(facecolor='#161b22', fontsize=9)
ax1.set_xlabel('Peso w', fontsize=11)
ax1.set_ylabel('Loss', fontsize=11)
ax1.set_title('Superfície de Erro 1D (Mínimos Locais vs Global)', color='#e6edf3', fontsize=11)
ax1.grid(True, alpha=0.3)

# ─── 3D: Superfície de erro em 2D de pesos ───────────────────────────────────
ax2 = fig.add_subplot(122, projection='3d')
ax2.set_facecolor('#0d1117')

w1 = np.linspace(-3, 3, 60)
w2 = np.linspace(-3, 3, 60)
W1, W2 = np.meshgrid(w1, w2)

# Loss surface sintética em 2D
Z = (np.sin(W1) * np.cos(W2) + 0.3 * (W1 ** 2 + W2 ** 2) * 0.4 + 0.2 * np.sin(2 * W1) * np.sin(2 * W2))
Z = Z - Z.min()

surf = ax2.plot_surface(W1, W2, Z, cmap='viridis', alpha=0.85, linewidth=0)

# Mínimo global
min_idx = np.unravel_index(np.argmin(Z), Z.shape)
ax2.scatter(W1[min_idx], W2[min_idx], Z[min_idx], color='#7ee787',
            s=200, marker='*', zorder=10, label='Mínimo Global')

ax2.set_xlabel('w₁', color='#8b949e', fontsize=9, labelpad=5)
ax2.set_ylabel('w₂', color='#8b949e', fontsize=9, labelpad=5)
ax2.set_zlabel('Loss', color='#8b949e', fontsize=9, labelpad=5)
ax2.set_title('Superfície de Erro 2D (espaço de pesos)', color='#e6edf3', fontsize=11)
ax2.tick_params(colors='#8b949e', labelsize=7)
ax2.legend(facecolor='#0d1117', edgecolor='#30363d', fontsize=9)

plt.tight_layout()
plt.show()

print("💡 Insight chave:")
print("   O Gradiente Descendente segue a inclinação LOCAL.")
print("   Sem estratégias especiais, pode ficar preso em mínimos locais.")
print("   Em redes neurais grandes, mínimos locais geralmente têm loss similar ao global.")


## ⚙️ 8. Comparando Otimizadores em PyTorch

Diferentes estratégias para navegar a superfície de erro e escapar de mínimos locais.


In [ ]:
# ─── Comparando Otimizadores ────────────────────────────────────────────────

torch.manual_seed(42)

resultados = {}

otimizadores_config = {
    'SGD (lr=0.01)': lambda p: torch.optim.SGD(p, lr=0.01),
    'SGD (lr=0.5)': lambda p: torch.optim.SGD(p, lr=0.5),
    'SGD + Momentum': lambda p: torch.optim.SGD(p, lr=0.1, momentum=0.9),
    'Adam': lambda p: torch.optim.Adam(p, lr=0.01),
}

for nome, opt_fn in otimizadores_config.items():
    torch.manual_seed(0)
    m = Perceptron(n_entradas=2)
    opt = opt_fn(m.parameters())
    hist = []
    for _ in range(300):
        pred = m(X)
        loss = criterio(pred, y_and)
        opt.zero_grad()
        loss.backward()
        opt.step()
        hist.append(loss.item())
    resultados[nome] = hist

# Visualização
fig, ax = plt.subplots(figsize=(11, 5))
cores = ['#ff7b72', '#ffa657', '#7ee787', '#79c0ff']

for (nome, hist), cor in zip(resultados.items(), cores):
    ax.plot(hist, color=cor, lw=2, label=f'{nome} → final: {hist[-1]:.4f}')

ax.set_xlabel('Época', fontsize=11)
ax.set_ylabel('BCE Loss', fontsize=11)
ax.set_title('Comparação de Otimizadores', color='#58a6ff', fontsize=14, fontweight='bold')
ax.legend(facecolor='#161b22', edgecolor='#30363d', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')
ax.set_ylabel('BCE Loss (escala log)', fontsize=11)

plt.tight_layout()
plt.show()

print("\n📊 Resumo dos otimizadores:")
print("─" * 50)
for nome, hist in resultados.items():
    print(f"  {nome:<22} → Loss final: {hist[-1]:.6f}")


## ✅ 9. Avaliando o Modelo Treinado

Vamos verificar se o nosso Perceptron aprendeu o AND lógico corretamente.


In [ ]:
# ─── Modelo Final e Fronteira de Decisão ───────────────────────────────────

# Retreinando com Adam para melhor convergência
torch.manual_seed(0)
modelo_final = Perceptron(n_entradas=2)
otimizador_final = torch.optim.Adam(modelo_final.parameters(), lr=0.1)

for _ in range(500):
    pred = modelo_final(X)
    loss = criterio(pred, y_and)
    otimizador_final.zero_grad()
    loss.backward()
    otimizador_final.step()

# Predições finais
with torch.no_grad():
    predicoes_finais = modelo_final(X)
    pred_binaria = (predicoes_finais >= 0.5).float()

print("✅ Predições APÓS o treinamento:")
print("─" * 55)
print(f"{'x1':>4} {'x2':>4} {'Esperado':>10} {'Prob':>10} {'Predito':>10} {'Correto':>10}")
print("─" * 55)
corretos = 0
for xi, yi, pi, pb in zip(X, y_and, pred_binaria, predicoes_finais):
    ok = "✅" if yi.item() == pi.item() else "❌"
    if yi.item() == pi.item():
        corretos += 1
    print(f"{xi[0].item():>4.0f} {xi[1].item():>4.0f} {yi.item():>10.0f} {pb.item():>10.4f} {pi.item():>10.0f} {ok:>10}")

acuracia = corretos / len(y_and) * 100
print(f"\n🎯 Acurácia: {corretos}/{len(y_and)} = {acuracia:.0f}%")

# ─── Fronteira de Decisão ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

# Grade de pontos para visualizar a fronteira
xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)

with torch.no_grad():
    zz = modelo_final(grid).reshape(xx.shape).numpy()

# Colorindo as regiões de decisão
contour = ax.contourf(xx, yy, zz, levels=50, cmap='RdYlGn', alpha=0.7, vmin=0, vmax=1)
ax.contour(xx, yy, zz, levels=[0.5], colors='white', linewidths=2, linestyles='--')
plt.colorbar(contour, ax=ax, label='Probabilidade predita')

# Pontos de dados
cores_pts = ['#ff7b72' if y == 0 else '#7ee787' for y in y_and.numpy().flatten()]
for xi, cor in zip(X.numpy(), cores_pts):
    ax.scatter(xi[0], xi[1], s=200, color=cor, edgecolors='white', lw=2, zorder=5)

ax.text(0, 0, 'AND=0', ha='center', va='bottom', fontsize=10, color='white')
ax.text(0, 1, 'AND=0', ha='center', va='bottom', fontsize=10, color='white')
ax.text(1, 0, 'AND=0', ha='center', va='bottom', fontsize=10, color='white')
ax.text(1, 1, 'AND=1', ha='center', va='bottom', fontsize=10, color='white')

patch_0 = mpatches.Patch(color='#ff7b72', label='Classe 0')
patch_1 = mpatches.Patch(color='#7ee787', label='Classe 1')
ax.legend(handles=[patch_0, patch_1], facecolor='#161b22', fontsize=10)
ax.set_xlabel('x₁', fontsize=12)
ax.set_ylabel('x₂', fontsize=12)
ax.set_title('Fronteira de Decisão — AND Lógico\n(linha tracejada = limiar 0.5)',
             color='#58a6ff', fontsize=12)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

print(f"\n🏆 Pesos finais aprendidos:")
print(f"   w₁ = {modelo_final.linear.weight.data[0, 0].item():+.4f}  (peso de x₁)")
print(f"   w₂ = {modelo_final.linear.weight.data[0, 1].item():+.4f}  (peso de x₂)")
print(f"   b  = {modelo_final.linear.bias.data.item():+.4f}  (intercepto)")


## 🗺️ 10. Resumo: O Mapa Completo do Perceptron

```
                    ┌──────────────────────────────────────────┐
                    │            PERCEPTRON                     │
                    │                                          │
  Entradas (x) ───► │  Sinapses (w, b)                         │
                    │       ↓                                  │
                    │  Soma Ponderada: z = w·x + b             │
                    │       ↓                                  │
                    │  Ativação: ŷ = f(z)         ────────────►│──► Predição (ŷ)
                    │                                          │
  Rótulos (y) ────► │  Função de Erro: L(ŷ, y)                │
                    │       ↓                                  │
                    │  Gradiente: ∂L/∂w                        │
                    │       ↓                                  │
                    │  Atualização: w ← w - η·∂L/∂w           │
                    └──────────────────────────────────────────┘
                              ↑ (repete até convergir)
```

### 📌 Essencial do PyTorch

```python
import torch
import torch.nn as nn

# 1. Definir o modelo
model = nn.Sequential(
    nn.Linear(n_entradas, 1),
    nn.Sigmoid()
)

# 2. Definir loss e otimizador
criterio   = nn.BCELoss()
otimizador = torch.optim.Adam(model.parameters(), lr=0.01)

# 3. Loop de treinamento
for epoca in range(n_epocas):
    y_pred = model(X)                  # Forward pass
    loss   = criterio(y_pred, y)       # Calcula erro
    otimizador.zero_grad()             # Zera gradientes
    loss.backward()                    # Backward pass
    otimizador.step()                  # Atualiza pesos
```

---

### 🚀 Próximos Passos

- **Multi-Layer Perceptron (MLP)** — empilhar camadas para resolver problemas não-lineares (XOR!)
- **Redes Convolucionais (CNN)** — para imagens
- **Backpropagation detalhado** — derivação completa do algoritmo
- **Regularização** — Dropout, L1/L2 para evitar overfitting
- **Batch Normalization** — estabilizar o treinamento

---
*Notebook criado inicialmente com a ferramenta claude.ai; está sendo constantemente atualizado no repositório.*
